# Music Generation Evaluation
Evaluates three GPT-2 models (base, Harte fine-tuned, ABC fine-tuned) using:
- **Jensen-Shannon Divergence (JSD)** — how well the model learned musical grammar vs. the training distribution
- **N-gram Novelty** — how original the generated chord progressions are

### 1. Imports

In [17]:
import re
import math
import json
import glob
import torch
from harte_to_abc_dict import harte_to_roman, abc_to_roman
from evaluations import jsd, ngram
from transformers import AutoTokenizer, AutoModelForCausalLM

### 2. Load Models

In [18]:
# Base GPT-2 (untouched)
base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
base_model     = AutoModelForCausalLM.from_pretrained("gpt2")

# Fine-tuned on Harte notation
harte_tokenizer = AutoTokenizer.from_pretrained("CS429/gpt2-harte")
harte_model     = AutoModelForCausalLM.from_pretrained("CS429/gpt2-harte")

# Fine-tuned on ABC notation
abc_tokenizer = AutoTokenizer.from_pretrained("CS429/gpt2-abc")
abc_model     = AutoModelForCausalLM.from_pretrained("CS429/gpt2-abc")

# Move to GPU if available
device = "mps" if torch.mps.is_available() else "cpu"
base_model.to(device)
harte_model.to(device)
abc_model.to(device)
print(f"Running on: {device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Running on: mps


### 3. Generation Helper

In [19]:
def generate(model, tokenizer, prompt, max_new_tokens=64):
    """
    Generate a completion from a model given a prompt string.
    Returns the full output string (prompt + generated continuation).
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

### 4. Tokenize Model Output into Chord Lists
After generating raw text, we need to parse it back into `list[str]` chord tokens before scoring.

In [20]:
def parse_harte_output(raw_text):
    """
    Extract Harte chord tokens from raw model output.
    Matches tokens like C:maj, F#:min7, Bb:hdim7, N, etc.
    Returns list[str].
    """
    pattern = r'[A-G][b#]?(?:b|#)?:[a-zA-Z0-9#b/()]+|N|X'
    return re.findall(pattern, raw_text)


def parse_abc_output(raw_text):
    """
    Extract ABC chord tokens from raw model output.
    Matches bracket groups like [C E G], [^F _B D], etc.
    Returns list[str].
    """
    return re.findall(r'\[[^\]]+\]', raw_text)

### 5. Load Datasets

In [21]:
# Maps ABC accidental key notation back to standard note names
ABC_KEY_MAP = {
    "_B": "Bb", "_E": "Eb", "_A": "Ab", "_D": "Db",
    "_G": "Gb", "^F": "F#", "^C": "C#", "^G": "G#",
    "^D": "D#", "^A": "A#", "_C": "Cb", "_F": "Fb",
    "^E": "E#", "^B": "B#"
}


def load_harte_dataset(data_dir="data/Harte"):
    """
    Returns:
        dataset_chords : list[list[str]]  raw Harte chord tokens per song
        dataset_rn     : list[list[str]]  Roman numeral tokens per song
        keys           : list[str]        key per song
    """
    dataset_chords, dataset_rn, keys = [], [], []

    for path in sorted(glob.glob(f"{data_dir}/*.json")):
        with open(path, "r", encoding="utf-8") as f:
            song = json.load(f)

        chords = song["chords"]
        key    = song["key"]
        rn     = harte_to_roman(key, chords)

        dataset_chords.append(chords)
        dataset_rn.append(rn)
        keys.append(key)

    print(f"Loaded {len(dataset_chords)} Harte songs from {data_dir}")
    return dataset_chords, dataset_rn, keys


def load_abc_dataset(data_dir="data/ABC"):
    """
    Returns:
        dataset_chords : list[list[str]]  raw ABC chord tokens per song
        dataset_rn     : list[list[str]]  Roman numeral tokens per song
        keys           : list[str]        key per song (converted to standard notation)
    """
    dataset_chords, dataset_rn, keys = [], [], []

    for path in sorted(glob.glob(f"{data_dir}/*.json")):
        with open(path, "r", encoding="utf-8") as f:
            song = json.load(f)

        chords = song["chords"]
        key    = ABC_KEY_MAP.get(song["key"], song["key"])  # normalise key
        rn     = abc_to_roman(key, chords)

        dataset_chords.append(chords)
        dataset_rn.append(rn)
        keys.append(key)

    print(f"Loaded {len(dataset_chords)} ABC songs from {data_dir}")
    return dataset_chords, dataset_rn, keys


harte_dataset_chords, harte_dataset_rn, harte_keys = load_harte_dataset("data/Harte")
abc_dataset_chords,   abc_dataset_rn,   abc_keys   = load_abc_dataset("data/ABC")

Loaded 890 Harte songs from data/Harte
Loaded 890 ABC songs from data/ABC


### 6. Prompts
Each prompt is a `(prompt_string, key)` pair. The key is needed to convert chords to Roman numerals for JSD. We use the same 250 musical ideas across all models.

In [22]:
# (prompt, key) — Harte notation
harte_prompts = [
    # ── original 20 ──────────────────────────────────────────────
    ("C:maj G:maj",                              "C"),
    ("A:min F:maj",                              "C"),
    ("D:maj A:maj",                              "D"),
    ("G:maj D:maj",                              "G"),
    ("F:maj C:maj",                              "F"),
    ("C:min G:min",                              "C"),
    ("E:min B:min",                              "G"),
    ("Bb:maj F:maj",                             "Bb"),
    ("A:maj F#:min",                             "A"),
    ("Ab:maj Eb:maj",                            "Ab"),
    ("C:maj A:min",                              "C"),
    ("D:min G:maj",                              "C"),
    ("B:min E:maj",                              "A"),
    ("F:min Bb:min",                             "Ab"),
    ("G:maj E:min",                              "G"),
    ("C:maj F:maj G:maj",                        "C"),
    ("A:min G:maj F:maj",                        "C"),
    ("D:maj G:maj A:maj",                        "D"),
    ("C:min Bb:maj",                             "C"),
    ("E:maj A:maj B:maj",                        "E"),
    # ── two-chord, more keys ──────────────────────────────────────
    ("G:maj C:maj",                              "G"),
    ("A:min E:min",                              "A"),
    ("F:maj Bb:maj",                             "F"),
    ("E:maj B:maj",                              "E"),
    ("G:min D:min",                              "Bb"),
    ("C#:min G#:min",                            "E"),
    ("Db:maj Ab:maj",                            "Db"),
    ("B:maj F#:maj",                             "B"),
    ("E:min A:min",                              "G"),
    ("F#:min B:min",                             "A"),
    # ── dominant 7ths ─────────────────────────────────────────────
    ("G:7 C:maj",                                "C"),
    ("D:7 G:maj",                                "G"),
    ("A:7 D:maj",                                "D"),
    ("E:7 A:maj",                                "A"),
    ("C:7 F:maj",                                "F"),
    ("F:7 Bb:maj",                               "Bb"),
    ("Bb:7 Eb:maj",                              "Eb"),
    ("Eb:7 Ab:maj",                              "Ab"),
    ("B:7 E:maj",                                "E"),
    ("F#:7 B:maj",                               "B"),
    # ── minor ii–V ────────────────────────────────────────────────
    ("D:min7 G:7",                               "C"),
    ("G:min7 C:7",                               "F"),
    ("A:min7 D:7",                               "G"),
    ("E:min7 A:7",                               "D"),
    ("B:min7 E:7",                               "A"),
    ("F#:min7 B:7",                              "E"),
    ("C:min7 F:7",                               "Bb"),
    ("F:min7 Bb:7",                              "Eb"),
    ("Bb:min7 Eb:7",                             "Ab"),
    ("C#:min7 F#:7",                             "B"),
    # ── major 7ths ────────────────────────────────────────────────
    ("C:maj7 F:maj",                             "C"),
    ("F:maj7 G:7",                               "C"),
    ("G:maj7 C:maj",                             "G"),
    ("D:maj7 G:maj",                             "D"),
    ("A:maj7 D:maj",                             "A"),
    ("E:maj7 A:maj",                             "E"),
    ("Bb:maj7 Eb:maj",                           "Eb"),
    ("Eb:maj7 Ab:maj",                           "Ab"),
    ("Ab:maj7 Db:maj",                           "Db"),
    ("B:maj7 E:maj",                             "B"),
    # ── three-chord progressions ──────────────────────────────────
    ("C:maj A:min F:maj",                        "C"),
    ("G:maj E:min C:maj",                        "G"),
    ("D:maj B:min G:maj",                        "D"),
    ("A:maj F#:min D:maj",                       "A"),
    ("E:maj C#:min A:maj",                       "E"),
    ("F:maj D:min Bb:maj",                       "F"),
    ("Bb:maj G:min Eb:maj",                      "Bb"),
    ("Eb:maj C:min Ab:maj",                      "Eb"),
    ("Ab:maj F:min Db:maj",                      "Ab"),
    ("C:min Ab:maj Eb:maj",                      "C"),
    ("G:min Eb:maj Bb:maj",                      "G"),
    ("D:min Bb:maj F:maj",                       "D"),
    ("A:min F:maj C:maj",                        "A"),
    ("E:min C:maj G:maj",                        "E"),
    ("B:min G:maj D:maj",                        "B"),
    # ── four-chord progressions ───────────────────────────────────
    ("C:maj F:maj G:7 C:maj",                    "C"),
    ("G:maj C:maj D:7 G:maj",                    "G"),
    ("D:maj G:maj A:7 D:maj",                    "D"),
    ("A:maj D:maj E:7 A:maj",                    "A"),
    ("F:maj Bb:maj C:7 F:maj",                   "F"),
    ("Bb:maj Eb:maj F:7 Bb:maj",                 "Bb"),
    ("C:maj A:min D:min G:7",                    "C"),
    ("G:maj E:min A:min D:7",                    "G"),
    ("D:maj B:min E:min A:7",                    "D"),
    ("A:maj F#:min B:min E:7",                   "A"),
    ("E:maj C#:min F#:min B:7",                  "E"),
    ("F:maj D:min G:min C:7",                    "F"),
    ("Bb:maj G:min C:min F:7",                   "Bb"),
    ("Eb:maj C:min F:min Bb:7",                  "Eb"),
    ("Ab:maj F:min Bb:min Eb:7",                 "Ab"),
    # ── mixed progressions ────────────────────────────────────────
    ("C:maj E:min F:maj G:maj",                  "C"),
    ("G:maj B:min C:maj D:maj",                  "G"),
    ("D:maj F#:min G:maj A:maj",                 "D"),
    ("A:maj C#:min D:maj E:maj",                 "A"),
    ("E:min D:maj C:maj G:maj",                  "E"),
    ("A:min G:maj F:maj E:7",                    "A"),
    ("D:min C:maj Bb:maj A:7",                   "D"),
    ("G:min F:maj Eb:maj D:7",                   "G"),
    ("C:min Bb:maj Ab:maj G:7",                  "C"),
    ("F:min Eb:maj Db:maj C:7",                  "F"),
    # ── I–ii two-chord across keys ────────────────────────────────
    ("C:maj D:min",                              "C"),
    ("G:maj A:min",                              "G"),
    ("D:maj E:min",                              "D"),
    ("A:maj B:min",                              "A"),
    ("E:maj F#:min",                             "E"),
    ("F:maj G:min",                              "F"),
    ("Bb:maj C:min",                             "Bb"),
    ("Eb:maj F:min",                             "Eb"),
    ("Ab:maj Bb:min",                            "Ab"),
    ("Db:maj Eb:min",                            "Db"),
    # ── bVII–I and iv–I colours ───────────────────────────────────
    ("G:min F:maj",                              "Bb"),
    ("D:min C:maj",                              "F"),
    ("A:min G:maj",                              "C"),
    ("E:min D:maj",                              "G"),
    ("B:min A:maj",                              "D"),
    ("F#:min E:maj",                             "A"),
    ("C#:min B:maj",                             "E"),
    ("Ab:maj Bb:7",                              "Eb"),
    ("Eb:maj F:7",                               "Bb"),
    ("Bb:maj C:7",                               "F"),
    # ── three-chord with 7ths ─────────────────────────────────────
    ("C:maj7 A:min7 D:min7",                     "C"),
    ("G:maj7 E:min7 A:min7",                     "G"),
    ("D:maj7 B:min7 E:min7",                     "D"),
    ("A:maj7 F#:min7 B:min7",                    "A"),
    ("F:maj7 D:min7 G:min7",                     "F"),
    ("Bb:maj7 G:min7 C:min7",                    "Bb"),
    ("Eb:maj7 C:min7 F:min7",                    "Eb"),
    ("D:min7 G:7 C:maj7",                        "C"),
    ("A:min7 D:7 G:maj7",                        "G"),
    ("E:min7 A:7 D:maj7",                        "D"),
    ("B:min7 E:7 A:maj7",                        "A"),
    ("F#:min7 B:7 E:maj7",                       "E"),
    ("G:min7 C:7 F:maj7",                        "F"),
    ("C:min7 F:7 Bb:maj7",                       "Bb"),
    ("F:min7 Bb:7 Eb:maj7",                      "Eb"),
    ("Bb:min7 Eb:7 Ab:maj7",                     "Ab"),
    ("F:maj7 E:min7 A:min7",                     "C"),
    ("G:7 F:maj7 C:maj",                         "C"),
    ("D:7 C:maj7 G:maj",                         "G"),
    ("A:7 G:maj7 D:maj",                         "D"),
    # ── I–V–IV–V riff patterns ────────────────────────────────────
    ("Eb:maj Bb:maj Ab:maj Bb:maj",              "Eb"),
    ("Ab:maj Eb:maj Db:maj Eb:maj",              "Ab"),
    ("Db:maj Ab:maj Gb:maj Ab:maj",              "Db"),
    ("F:maj C:maj Bb:maj C:maj",                 "F"),
    ("Bb:maj F:maj Eb:maj F:maj",                "Bb"),
    ("E:maj B:maj A:maj B:maj",                  "E"),
    ("B:maj F#:maj E:maj F#:maj",                "B"),
    ("D:maj A:maj G:maj A:maj",                  "D"),
    ("A:maj E:maj D:maj E:maj",                  "A"),
    ("G:maj D:maj C:maj D:maj",                  "G"),
    # ── i–v oscillations and minor tonic expansions ───────────────
    ("C:min G:min F:maj G:min",                  "C"),
    ("G:min D:min C:maj D:min",                  "G"),
    ("D:min A:min G:maj A:min",                  "D"),
    ("A:min E:min D:maj E:min",                  "A"),
    ("E:min B:min A:maj B:min",                  "E"),
    ("F:maj G:maj A:min E:min",                  "C"),
    ("Bb:maj C:maj D:min A:min",                 "F"),
    ("Eb:maj F:maj G:min D:min",                 "Bb"),
    ("Ab:maj Bb:maj C:min G:min",                "Eb"),
    ("Db:maj Eb:maj F:min C:min",                "Ab"),
    # ── natural minor i–bVI–bIII–bVII ────────────────────────────
    ("A:min F:maj C:maj G:maj",                  "A"),
    ("E:min C:maj G:maj D:maj",                  "E"),
    ("B:min G:maj D:maj A:maj",                  "B"),
    ("F#:min D:maj A:maj E:maj",                 "F#"),
    ("D:min Bb:maj F:maj C:maj",                 "D"),
    ("G:min Eb:maj Bb:maj F:maj",                "G"),
    ("C:min Ab:maj Eb:maj Bb:maj",               "C"),
    ("F:min Db:maj Ab:maj Eb:maj",               "F"),
    # ── harmonic minor (raised V7) ────────────────────────────────
    ("A:min E:7 F:maj G:7",                      "A"),
    ("E:min B:7 C:maj D:7",                      "E"),
    ("D:min A:7 Bb:maj C:7",                     "D"),
    ("G:min D:7 Eb:maj F:7",                     "G"),
    ("C:min G:7 Ab:maj Bb:7",                    "C"),
    ("F:min C:7 Db:maj Eb:7",                    "F"),
    ("A:min C:maj D:min E:7",                    "A"),
    ("E:min G:maj A:min B:7",                    "E"),
    ("D:min F:maj G:min A:7",                    "D"),
    ("G:min Bb:maj C:min D:7",                   "G"),
    ("C:min Eb:maj F:min G:7",                   "C"),
    ("B:min D:maj E:min F#:7",                   "B"),
    # ── full ii7–V7–Imaj7–IVmaj7 ─────────────────────────────────
    ("D:min7 G:7 C:maj7 F:maj7",                 "C"),
    ("A:min7 D:7 G:maj7 C:maj7",                 "G"),
    ("E:min7 A:7 D:maj7 G:maj7",                 "D"),
    ("B:min7 E:7 A:maj7 D:maj7",                 "A"),
    ("F#:min7 B:7 E:maj7 A:maj7",                "E"),
    ("G:min7 C:7 F:maj7 Bb:maj7",                "F"),
    ("C:min7 F:7 Bb:maj7 Eb:maj7",               "Bb"),
    ("F:min7 Bb:7 Eb:maj7 Ab:maj7",              "Eb"),
    ("Bb:min7 Eb:7 Ab:maj7 Db:maj7",             "Ab"),
    # ── diatonic 7th stacks Imaj7–ii7–iii7–IVmaj7 ────────────────
    ("C:maj7 D:min7 E:min7 F:maj7",              "C"),
    ("G:maj7 A:min7 B:min7 C:maj7",              "G"),
    ("D:maj7 E:min7 F#:min7 G:maj7",             "D"),
    ("A:maj7 B:min7 C#:min7 D:maj7",             "A"),
    ("F:maj7 G:min7 A:min7 Bb:maj7",             "F"),
    ("Bb:maj7 C:min7 D:min7 Eb:maj7",            "Bb"),
    # ── dominant circle ───────────────────────────────────────────
    ("C:7 F:7 Bb:7 Eb:7",                        "Eb"),
    ("G:7 C:7 F:7 Bb:7",                         "Bb"),
    ("D:7 G:7 C:7 F:7",                          "F"),
    ("A:7 D:7 G:7 C:7",                          "C"),
    ("E:7 A:7 D:7 G:7",                          "G"),
    # ── five-chord I–vi–IV–V–I ────────────────────────────────────
    ("C:maj A:min F:maj G:maj C:maj",            "C"),
    ("G:maj E:min C:maj D:maj G:maj",            "G"),
    ("D:maj B:min G:maj A:maj D:maj",            "D"),
    ("A:maj F#:min D:maj E:maj A:maj",           "A"),
    ("F:maj D:min Bb:maj C:maj F:maj",           "F"),
    ("Bb:maj G:min Eb:maj F:maj Bb:maj",         "Bb"),
    # ── five-chord I–ii–iii–IV–V ──────────────────────────────────
    ("C:maj D:min E:min F:maj G:maj",            "C"),
    ("G:maj A:min B:min C:maj D:maj",            "G"),
    ("D:maj E:min F#:min G:maj A:maj",           "D"),
    # ── five-chord I–IV–I–V–I ─────────────────────────────────────
    ("C:maj F:maj C:maj G:maj C:maj",            "C"),
    ("G:maj C:maj G:maj D:maj G:maj",            "G"),
    # ── five-chord with 7ths ──────────────────────────────────────
    ("D:min7 G:7 C:maj F:maj G:7",               "C"),
    ("A:min7 D:7 G:maj C:maj D:7",               "G"),
    ("E:min G:maj A:min C:maj D:maj",            "G"),
    ("A:min C:maj D:min F:maj E:7",              "A"),
    # ── I–V–vi–IV and variants (five chords) ─────────────────────
    ("C:maj G:maj A:min F:maj C:maj",            "C"),
    ("G:maj D:maj E:min C:maj G:maj",            "G"),
    ("F:maj C:maj D:min Bb:maj F:maj",           "F"),
    ("Bb:maj F:maj G:min Eb:maj Bb:maj",         "Bb"),
    ("Eb:maj Bb:maj C:min Ab:maj Eb:maj",        "Eb"),
    ("Ab:maj Eb:maj F:min Db:maj Ab:maj",        "Ab"),
    # ── borrowed bIII/bVII chords ─────────────────────────────────
    ("C:min Bb:maj Ab:maj Bb:maj C:min",         "C"),
    ("G:min F:maj Eb:maj F:maj G:min",           "G"),
    ("D:min C:maj Bb:maj C:maj D:min",           "D"),
    ("A:min G:maj F:maj G:maj A:min",            "A"),
    ("C:maj Eb:maj Bb:maj F:maj",                "C"),
    ("G:maj Bb:maj F:maj C:maj",                 "G"),
    ("D:maj F:maj C:maj G:maj",                  "D"),
    ("A:maj C:maj G:maj D:maj",                  "A"),
    ("E:maj G:maj D:maj A:maj",                  "E"),
    # ── I–IV–I–bVII ───────────────────────────────────────────────
    ("C:maj F:maj C:maj Bb:maj",                 "C"),
    ("G:maj C:maj G:maj F:maj",                  "G"),
    ("D:maj G:maj D:maj C:maj",                  "D"),
    ("A:maj D:maj A:maj G:maj",                  "A"),
    ("E:maj A:maj E:maj D:maj",                  "E"),
    # ── harmonic minor cadences i–iv–V7–i ────────────────────────
    ("C:min F:min G:7 C:min",                    "C"),
    ("G:min C:min D:7 G:min",                    "G"),
    ("D:min G:min A:7 D:min",                    "D"),
    ("A:min D:min E:7 A:min",                    "A"),
    ("E:min A:min B:7 E:min",                    "E"),
    # ── descending bass / I–V–IV–iii ─────────────────────────────
    ("C:maj G:maj F:maj E:min",                  "C"),
    ("G:maj D:maj C:maj B:min",                  "G"),
    ("D:maj A:maj G:maj F#:min",                 "D"),
    ("A:maj E:maj D:maj C#:min",                 "A"),
    ("F:maj C:maj Bb:maj A:min",                 "F"),
    ("Bb:maj F:maj Eb:maj D:min",                "Bb"),
    ("Eb:maj Bb:maj Ab:maj G:min",               "Eb"),
    ("Ab:maj Eb:maj Db:maj C:min",               "Ab"),
    # ── Imaj7–V7–vi7–IVmaj7 jazz turnarounds ─────────────────────
    ("C:maj7 G:7 A:min7 F:maj7",                 "C"),
    ("G:maj7 D:7 E:min7 C:maj7",                 "G"),
]

# Same musical ideas in ABC notation (key column is identical)
abc_prompts = [
    # ── original 20 ──────────────────────────────────────────────
    ("[C E G] [G B D]",                                   "C"),
    ("[A C E] [F A C]",                                   "C"),
    ("[D ^F A] [A ^C E]",                                 "D"),
    ("[G B D] [D ^F A]",                                  "G"),
    ("[F A C] [C E G]",                                   "F"),
    ("[C _E G] [G _B D]",                                 "C"),
    ("[E G B] [B ^D ^F]",                                 "G"),
    ("[_B D F] [F A C]",                                  "Bb"),
    ("[A ^C E] [^F A ^C]",                                "A"),
    ("[_A C _E] [_E G _B]",                               "Ab"),
    ("[C E G] [A C E]",                                   "C"),
    ("[D F A] [G B D]",                                   "C"),
    ("[B ^D ^F] [E ^G B]",                                "A"),
    ("[F _A C] [_B _D F]",                                "Ab"),
    ("[G B D] [E G B]",                                   "G"),
    ("[C E G] [F A C] [G B D]",                           "C"),
    ("[A C E] [G B D] [F A C]",                           "C"),
    ("[D ^F A] [G B D] [A ^C E]",                         "D"),
    ("[C _E G] [_B D F]",                                 "C"),
    ("[E ^G B] [A ^C E] [B ^D ^F]",                       "E"),
    # ── two-chord, more keys ──────────────────────────────────────
    ("[G B D] [C E G]",                                   "G"),
    ("[A C E] [E G B]",                                   "A"),
    ("[F A C] [_B D F]",                                  "F"),
    ("[E ^G B] [B ^D ^F]",                                "E"),
    ("[G _B D] [D F A]",                                  "Bb"),
    ("[^C E ^G] [^G B ^D]",                               "E"),
    ("[_D F _A] [_A C _E]",                               "Db"),
    ("[B ^D ^F] [^F ^A ^C]",                              "B"),
    ("[E G B] [A C E]",                                   "G"),
    ("[^F A ^C] [B D ^F]",                                "A"),
    # ── dominant 7ths ─────────────────────────────────────────────
    ("[G B D F] [C E G]",                                 "C"),
    ("[D ^F A C] [G B D]",                                "G"),
    ("[A ^C E G] [D ^F A]",                               "D"),
    ("[E ^G B D] [A ^C E]",                               "A"),
    ("[C E G _B] [F A C]",                                "F"),
    ("[F A C _E] [_B D F]",                               "Bb"),
    ("[_B D F _A] [_E G _B]",                             "Eb"),
    ("[_E G _B _D] [_A C _E]",                            "Ab"),
    ("[B ^D ^F A] [E ^G B]",                              "E"),
    ("[^F ^A ^C E] [B ^D ^F]",                            "B"),
    # ── minor ii–V ────────────────────────────────────────────────
    ("[D F A C] [G B D F]",                               "C"),
    ("[G _B D F] [C E G _B]",                             "F"),
    ("[A C E G] [D ^F A C]",                              "G"),
    ("[E G B D] [A ^C E G]",                              "D"),
    ("[B D ^F A] [E ^G B D]",                             "A"),
    ("[^F A ^C E] [B ^D ^F A]",                           "E"),
    ("[C _E G _B] [F A C _E]",                            "Bb"),
    ("[F _A C _E] [_B D F _A]",                           "Eb"),
    ("[_B _D F _A] [_E G _B _D]",                         "Ab"),
    ("[^C E ^G B] [^F ^A ^C E]",                          "B"),
    # ── major 7ths ────────────────────────────────────────────────
    ("[C E G B] [F A C]",                                 "C"),
    ("[F A C E] [G B D F]",                               "C"),
    ("[G B D ^F] [C E G]",                                "G"),
    ("[D ^F A ^C] [G B D]",                               "D"),
    ("[A ^C E ^G] [D ^F A]",                              "A"),
    ("[E ^G B ^D] [A ^C E]",                              "E"),
    ("[_B D F A] [_E G _B]",                              "Eb"),
    ("[_E G _B D] [_A C _E]",                             "Ab"),
    ("[_A C _E G] [_D F _A]",                             "Db"),
    ("[B ^D ^F ^A] [E ^G B]",                             "B"),
    # ── three-chord progressions ──────────────────────────────────
    ("[C E G] [A C E] [F A C]",                           "C"),
    ("[G B D] [E G B] [C E G]",                           "G"),
    ("[D ^F A] [B D ^F] [G B D]",                         "D"),
    ("[A ^C E] [^F A ^C] [D ^F A]",                       "A"),
    ("[E ^G B] [^C E ^G] [A ^C E]",                       "E"),
    ("[F A C] [D F A] [_B D F]",                          "F"),
    ("[_B D F] [G _B D] [_E G _B]",                       "Bb"),
    ("[_E G _B] [C _E G] [_A C _E]",                      "Eb"),
    ("[_A C _E] [F _A C] [_D F _A]",                      "Ab"),
    ("[C _E G] [_A C _E] [_E G _B]",                      "C"),
    ("[G _B D] [_E G _B] [_B D F]",                       "G"),
    ("[D F A] [_B D F] [F A C]",                          "D"),
    ("[A C E] [F A C] [C E G]",                           "A"),
    ("[E G B] [C E G] [G B D]",                           "E"),
    ("[B D ^F] [G B D] [D ^F A]",                         "B"),
    # ── four-chord progressions ───────────────────────────────────
    ("[C E G] [F A C] [G B D F] [C E G]",                 "C"),
    ("[G B D] [C E G] [D ^F A C] [G B D]",                "G"),
    ("[D ^F A] [G B D] [A ^C E G] [D ^F A]",              "D"),
    ("[A ^C E] [D ^F A] [E ^G B D] [A ^C E]",             "A"),
    ("[F A C] [_B D F] [C E G _B] [F A C]",               "F"),
    ("[_B D F] [_E G _B] [F A C _E] [_B D F]",            "Bb"),
    ("[C E G] [A C E] [D F A] [G B D F]",                 "C"),
    ("[G B D] [E G B] [A C E] [D ^F A C]",                "G"),
    ("[D ^F A] [B D ^F] [E G B] [A ^C E G]",              "D"),
    ("[A ^C E] [^F A ^C] [B D ^F] [E ^G B D]",            "A"),
    ("[E ^G B] [^C E ^G] [^F A ^C] [B ^D ^F A]",          "E"),
    ("[F A C] [D F A] [G _B D] [C E G _B]",               "F"),
    ("[_B D F] [G _B D] [C _E G] [F A C _E]",             "Bb"),
    ("[_E G _B] [C _E G] [F _A C] [_B D F _A]",           "Eb"),
    ("[_A C _E] [F _A C] [_B _D F] [_E G _B _D]",         "Ab"),
    # ── mixed progressions ────────────────────────────────────────
    ("[C E G] [E G B] [F A C] [G B D]",                   "C"),
    ("[G B D] [B D ^F] [C E G] [D ^F A]",                 "G"),
    ("[D ^F A] [^F A ^C] [G B D] [A ^C E]",               "D"),
    ("[A ^C E] [^C E ^G] [D ^F A] [E ^G B]",              "A"),
    ("[E G B] [D ^F A] [C E G] [G B D]",                  "E"),
    ("[A C E] [G B D] [F A C] [E ^G B D]",                "A"),
    ("[D F A] [C E G] [_B D F] [A ^C E G]",               "D"),
    ("[G _B D] [F A C] [_E G _B] [D ^F A C]",             "G"),
    ("[C _E G] [_B D F] [_A C _E] [G B D F]",             "C"),
    ("[F _A C] [_E G _B] [_D F _A] [C E G _B]",           "F"),
    # ── I–ii two-chord across keys ────────────────────────────────
    ("[C E G] [D F A]",                                   "C"),
    ("[G B D] [A C E]",                                   "G"),
    ("[D ^F A] [E G B]",                                  "D"),
    ("[A ^C E] [B D ^F]",                                 "A"),
    ("[E ^G B] [^F A ^C]",                                "E"),
    ("[F A C] [G _B D]",                                  "F"),
    ("[_B D F] [C _E G]",                                 "Bb"),
    ("[_E G _B] [F _A C]",                                "Eb"),
    ("[_A C _E] [_B _D F]",                               "Ab"),
    ("[_D F _A] [_E _G _B]",                              "Db"),
    # ── bVII–I and iv–I colours ───────────────────────────────────
    ("[G _B D] [F A C]",                                  "Bb"),
    ("[D F A] [C E G]",                                   "F"),
    ("[A C E] [G B D]",                                   "C"),
    ("[E G B] [D ^F A]",                                  "G"),
    ("[B D ^F] [A ^C E]",                                 "D"),
    ("[^F A ^C] [E ^G B]",                                "A"),
    ("[^C E ^G] [B ^D ^F]",                               "E"),
    ("[_A C _E] [_B D F _A]",                             "Eb"),
    ("[_E G _B] [F A C _E]",                              "Bb"),
    ("[_B D F] [C E G _B]",                               "F"),
    # ── three-chord with 7ths ─────────────────────────────────────
    ("[C E G B] [A C E G] [D F A C]",                     "C"),
    ("[G B D ^F] [E G B D] [A C E G]",                    "G"),
    ("[D ^F A ^C] [B D ^F A] [E G B D]",                  "D"),
    ("[A ^C E ^G] [^F A ^C E] [B D ^F A]",                "A"),
    ("[F A C E] [D F A C] [G _B D F]",                    "F"),
    ("[_B D F A] [G _B D F] [C _E G _B]",                 "Bb"),
    ("[_E G _B D] [C _E G _B] [F _A C _E]",               "Eb"),
    ("[D F A C] [G B D F] [C E G B]",                     "C"),
    ("[A C E G] [D ^F A C] [G B D ^F]",                   "G"),
    ("[E G B D] [A ^C E G] [D ^F A ^C]",                  "D"),
    ("[B D ^F A] [E ^G B D] [A ^C E ^G]",                 "A"),
    ("[^F A ^C E] [B ^D ^F A] [E ^G B ^D]",               "E"),
    ("[G _B D F] [C E G _B] [F A C E]",                   "F"),
    ("[C _E G _B] [F A C _E] [_B D F A]",                 "Bb"),
    ("[F _A C _E] [_B D F _A] [_E G _B D]",               "Eb"),
    ("[_B _D F _A] [_E G _B _D] [_A C _E G]",             "Ab"),
    ("[F A C E] [E G B D] [A C E G]",                     "C"),
    ("[G B D F] [F A C E] [C E G]",                       "C"),
    ("[D ^F A C] [C E G B] [G B D]",                      "G"),
    ("[A ^C E G] [G B D ^F] [D ^F A]",                    "D"),
    # ── I–V–IV–V riff patterns ────────────────────────────────────
    ("[_E G _B] [_B D F] [_A C _E] [_B D F]",             "Eb"),
    ("[_A C _E] [_E G _B] [_D F _A] [_E G _B]",           "Ab"),
    ("[_D F _A] [_A C _E] [_G _B _D] [_A C _E]",          "Db"),
    ("[F A C] [C E G] [_B D F] [C E G]",                  "F"),
    ("[_B D F] [F A C] [_E G _B] [F A C]",                "Bb"),
    ("[E ^G B] [B ^D ^F] [A ^C E] [B ^D ^F]",             "E"),
    ("[B ^D ^F] [^F ^A ^C] [E ^G B] [^F ^A ^C]",          "B"),
    ("[D ^F A] [A ^C E] [G B D] [A ^C E]",                "D"),
    ("[A ^C E] [E ^G B] [D ^F A] [E ^G B]",               "A"),
    ("[G B D] [D ^F A] [C E G] [D ^F A]",                 "G"),
    # ── i–v oscillations and minor tonic expansions ───────────────
    ("[C _E G] [G _B D] [F A C] [G _B D]",                "C"),
    ("[G _B D] [D F A] [C E G] [D F A]",                  "G"),
    ("[D F A] [A C E] [G B D] [A C E]",                   "D"),
    ("[A C E] [E G B] [D ^F A] [E G B]",                  "A"),
    ("[E G B] [B D ^F] [A ^C E] [B D ^F]",                "E"),
    ("[F A C] [G B D] [A C E] [E G B]",                   "C"),
    ("[_B D F] [C E G] [D F A] [A C E]",                  "F"),
    ("[_E G _B] [F A C] [G _B D] [D F A]",                "Bb"),
    ("[_A C _E] [_B D F] [C _E G] [G _B D]",              "Eb"),
    ("[_D F _A] [_E G _B] [F _A C] [C _E G]",             "Ab"),
    # ── natural minor i–bVI–bIII–bVII ────────────────────────────
    ("[A C E] [F A C] [C E G] [G B D]",                   "A"),
    ("[E G B] [C E G] [G B D] [D ^F A]",                  "E"),
    ("[B D ^F] [G B D] [D ^F A] [A ^C E]",                "B"),
    ("[^F A ^C] [D ^F A] [A ^C E] [E ^G B]",              "F#"),
    ("[D F A] [_B D F] [F A C] [C E G]",                  "D"),
    ("[G _B D] [_E G _B] [_B D F] [F A C]",               "G"),
    ("[C _E G] [_A C _E] [_E G _B] [_B D F]",             "C"),
    ("[F _A C] [_D F _A] [_A C _E] [_E G _B]",            "F"),
    # ── harmonic minor (raised V7) ────────────────────────────────
    ("[A C E] [E ^G B D] [F A C] [G B D F]",              "A"),
    ("[E G B] [B ^D ^F A] [C E G] [D ^F A C]",            "E"),
    ("[D F A] [A ^C E G] [_B D F] [C E G _B]",            "D"),
    ("[G _B D] [D ^F A C] [_E G _B] [F A C _E]",          "G"),
    ("[C _E G] [G B D F] [_A C _E] [_B D F _A]",          "C"),
    ("[F _A C] [C E G _B] [_D F _A] [_E G _B _D]",        "F"),
    ("[A C E] [C E G] [D F A] [E ^G B D]",                "A"),
    ("[E G B] [G B D] [A C E] [B ^D ^F A]",               "E"),
    ("[D F A] [F A C] [G _B D] [A ^C E G]",               "D"),
    ("[G _B D] [_B D F] [C _E G] [D ^F A C]",             "G"),
    ("[C _E G] [_E G _B] [F _A C] [G B D F]",             "C"),
    ("[B D ^F] [D ^F A] [E G B] [^F ^A ^C E]",            "B"),
    # ── full ii7–V7–Imaj7–IVmaj7 ─────────────────────────────────
    ("[D F A C] [G B D F] [C E G B] [F A C E]",           "C"),
    ("[A C E G] [D ^F A C] [G B D ^F] [C E G B]",         "G"),
    ("[E G B D] [A ^C E G] [D ^F A ^C] [G B D ^F]",       "D"),
    ("[B D ^F A] [E ^G B D] [A ^C E ^G] [D ^F A ^C]",     "A"),
    ("[^F A ^C E] [B ^D ^F A] [E ^G B ^D] [A ^C E ^G]",   "E"),
    ("[G _B D F] [C E G _B] [F A C E] [_B D F A]",        "F"),
    ("[C _E G _B] [F A C _E] [_B D F A] [_E G _B D]",     "Bb"),
    ("[F _A C _E] [_B D F _A] [_E G _B D] [_A C _E G]",   "Eb"),
    ("[_B _D F _A] [_E G _B _D] [_A C _E G] [_D F _A C]", "Ab"),
    # ── diatonic 7th stacks Imaj7–ii7–iii7–IVmaj7 ────────────────
    ("[C E G B] [D F A C] [E G B D] [F A C E]",           "C"),
    ("[G B D ^F] [A C E G] [B D ^F A] [C E G B]",         "G"),
    ("[D ^F A ^C] [E G B D] [^F A ^C E] [G B D ^F]",      "D"),
    ("[A ^C E ^G] [B D ^F A] [^C E ^G B] [D ^F A ^C]",    "A"),
    ("[F A C E] [G _B D F] [A C E G] [_B D F A]",         "F"),
    ("[_B D F A] [C _E G _B] [D F A C] [_E G _B D]",      "Bb"),
    # ── dominant circle ───────────────────────────────────────────
    ("[C E G _B] [F A C _E] [_B D F _A] [_E G _B _D]",    "Eb"),
    ("[G B D F] [C E G _B] [F A C _E] [_B D F _A]",       "Bb"),
    ("[D ^F A C] [G B D F] [C E G _B] [F A C _E]",        "F"),
    ("[A ^C E G] [D ^F A C] [G B D F] [C E G _B]",        "C"),
    ("[E ^G B D] [A ^C E G] [D ^F A C] [G B D F]",        "G"),
    # ── five-chord I–vi–IV–V–I ────────────────────────────────────
    ("[C E G] [A C E] [F A C] [G B D] [C E G]",           "C"),
    ("[G B D] [E G B] [C E G] [D ^F A] [G B D]",          "G"),
    ("[D ^F A] [B D ^F] [G B D] [A ^C E] [D ^F A]",       "D"),
    ("[A ^C E] [^F A ^C] [D ^F A] [E ^G B] [A ^C E]",     "A"),
    ("[F A C] [D F A] [_B D F] [C E G] [F A C]",          "F"),
    ("[_B D F] [G _B D] [_E G _B] [F A C] [_B D F]",      "Bb"),
    # ── five-chord I–ii–iii–IV–V ──────────────────────────────────
    ("[C E G] [D F A] [E G B] [F A C] [G B D]",           "C"),
    ("[G B D] [A C E] [B D ^F] [C E G] [D ^F A]",         "G"),
    ("[D ^F A] [E G B] [^F A ^C] [G B D] [A ^C E]",       "D"),
    # ── five-chord I–IV–I–V–I ─────────────────────────────────────
    ("[C E G] [F A C] [C E G] [G B D] [C E G]",           "C"),
    ("[G B D] [C E G] [G B D] [D ^F A] [G B D]",          "G"),
    # ── five-chord with 7ths ──────────────────────────────────────
    ("[D F A C] [G B D F] [C E G] [F A C] [G B D F]",     "C"),
    ("[A C E G] [D ^F A C] [G B D] [C E G] [D ^F A C]",   "G"),
    ("[E G B] [G B D] [A C E] [C E G] [D ^F A]",          "G"),
    ("[A C E] [C E G] [D F A] [F A C] [E ^G B D]",        "A"),
    # ── I–V–vi–IV and variants ────────────────────────────────────
    ("[C E G] [G B D] [A C E] [F A C] [C E G]",           "C"),
    ("[G B D] [D ^F A] [E G B] [C E G] [G B D]",          "G"),
    ("[F A C] [C E G] [D F A] [_B D F] [F A C]",          "F"),
    ("[_B D F] [F A C] [G _B D] [_E G _B] [_B D F]",      "Bb"),
    ("[_E G _B] [_B D F] [C _E G] [_A C _E] [_E G _B]",   "Eb"),
    ("[_A C _E] [_E G _B] [F _A C] [_D F _A] [_A C _E]",  "Ab"),
    # ── borrowed bIII/bVII chords ─────────────────────────────────
    ("[C _E G] [_B D F] [_A C _E] [_B D F] [C _E G]",     "C"),
    ("[G _B D] [F A C] [_E G _B] [F A C] [G _B D]",       "G"),
    ("[D F A] [C E G] [_B D F] [C E G] [D F A]",          "D"),
    ("[A C E] [G B D] [F A C] [G B D] [A C E]",           "A"),
    ("[C E G] [_E G _B] [_B D F] [F A C]",                "C"),
    ("[G B D] [_B D F] [F A C] [C E G]",                  "G"),
    ("[D ^F A] [F A C] [C E G] [G B D]",                  "D"),
    ("[A ^C E] [C E G] [G B D] [D ^F A]",                 "A"),
    ("[E ^G B] [G B D] [D ^F A] [A ^C E]",                "E"),
    # ── I–IV–I–bVII ───────────────────────────────────────────────
    ("[C E G] [F A C] [C E G] [_B D F]",                  "C"),
    ("[G B D] [C E G] [G B D] [F A C]",                   "G"),
    ("[D ^F A] [G B D] [D ^F A] [C E G]",                 "D"),
    ("[A ^C E] [D ^F A] [A ^C E] [G B D]",                "A"),
    ("[E ^G B] [A ^C E] [E ^G B] [D ^F A]",               "E"),
    # ── harmonic minor cadences i–iv–V7–i ────────────────────────
    ("[C _E G] [F _A C] [G B D F] [C _E G]",              "C"),
    ("[G _B D] [C _E G] [D ^F A C] [G _B D]",             "G"),
    ("[D F A] [G _B D] [A ^C E G] [D F A]",               "D"),
    ("[A C E] [D F A] [E ^G B D] [A C E]",                "A"),
    ("[E G B] [A C E] [B ^D ^F A] [E G B]",               "E"),
    # ── descending bass / I–V–IV–iii ─────────────────────────────
    ("[C E G] [G B D] [F A C] [E G B]",                   "C"),
    ("[G B D] [D ^F A] [C E G] [B D ^F]",                 "G"),
    ("[D ^F A] [A ^C E] [G B D] [^F A ^C]",               "D"),
    ("[A ^C E] [E ^G B] [D ^F A] [^C E ^G]",              "A"),
    ("[F A C] [C E G] [_B D F] [A C E]",                  "F"),
    ("[_B D F] [F A C] [_E G _B] [D F A]",                "Bb"),
    ("[_E G _B] [_B D F] [_A C _E] [G _B D]",             "Eb"),
    ("[_A C _E] [_E G _B] [_D F _A] [C _E G]",            "Ab"),
    # ── Imaj7–V7–vi7–IVmaj7 jazz turnarounds ─────────────────────
    ("[C E G B] [G B D F] [A C E G] [F A C E]",           "C"),
    ("[G B D ^F] [D ^F A C] [E G B D] [C E G B]",         "G"),
]

### 7. Run Evaluation Loop

In [23]:
def evaluate_model(
    model, tokenizer,
    prompts,            # list of (prompt_str, key)
    parse_fn,           # parse_harte_output or parse_abc_output
    to_rn_fn,           # harte_to_roman or abc_to_roman
    dataset_chords,     # list[list[str]] — raw chord dataset for n-gram
    dataset_rn,         # list[list[str]] — RN dataset for JSD
    model_name="model",
):
    all_jsd, all_bi, all_tri, all_quad = [], [], [], []

    for i, (prompt, key) in enumerate(prompts):
        # 1. Generate
        raw = generate(model, tokenizer, prompt)

        # 2. Parse into chord token list
        progression = parse_fn(raw)

        if len(progression) < 2:
            print(f"  [{model_name}] Prompt {i+1}: too few chords parsed, skipping.")
            continue

        # 3. Convert to Roman numerals for JSD
        progression_rn = to_rn_fn(key, progression)

        if not progression_rn:
            print(f"  [{model_name}] Prompt {i+1}: RN conversion empty, skipping.")
            continue

        # 4. Score
        all_jsd.append(jsd(progression_rn, dataset_rn))
        all_bi.append(ngram(progression, dataset_chords, n=2))
        all_tri.append(ngram(progression, dataset_chords, n=3))
        all_quad.append(ngram(progression, dataset_chords, n=4))

        print(f"  [{model_name}] Prompt {i+1}/{len(prompts)} done — "
              f"JSD={all_jsd[-1]:.4f}, "
              f"2g={all_bi[-1]:.4f}, "
              f"3g={all_tri[-1]:.4f}, "
              f"4g={all_quad[-1]:.4f}")

    def mean(lst): return sum(lst) / len(lst) if lst else float("nan")

    return {
        "model":            model_name,
        "jsd_mean":         mean(all_jsd),
        "bigram_novelty":   mean(all_bi),
        "trigram_novelty":  mean(all_tri),
        "fourgram_novelty": mean(all_quad),
        "n_scored":         len(all_jsd),
    }


print("=== Base GPT-2 (Harte prompts) ===")
base_harte_results = evaluate_model(
    base_model, base_tokenizer,
    prompts        = harte_prompts,
    parse_fn       = parse_harte_output,
    to_rn_fn       = harte_to_roman,
    dataset_chords = harte_dataset_chords,
    dataset_rn     = harte_dataset_rn,
    model_name     = "base-harte",
)

print("\n=== Base GPT-2 (ABC prompts) ===")
base_abc_results = evaluate_model(
    base_model, base_tokenizer,
    prompts        = abc_prompts,
    parse_fn       = parse_abc_output,
    to_rn_fn       = abc_to_roman,
    dataset_chords = abc_dataset_chords,
    dataset_rn     = abc_dataset_rn,
    model_name     = "base-abc",
)

print("\n=== Harte Fine-tuned ===")
harte_results = evaluate_model(
    harte_model, harte_tokenizer,
    prompts        = harte_prompts,
    parse_fn       = parse_harte_output,
    to_rn_fn       = harte_to_roman,
    dataset_chords = harte_dataset_chords,
    dataset_rn     = harte_dataset_rn,
    model_name     = "harte",
)

print("\n=== ABC Fine-tuned ===")
abc_results = evaluate_model(
    abc_model, abc_tokenizer,
    prompts        = abc_prompts,
    parse_fn       = parse_abc_output,
    to_rn_fn       = abc_to_roman,
    dataset_chords = abc_dataset_chords,
    dataset_rn     = abc_dataset_rn,
    model_name     = "abc",
)

=== Base GPT-2 (Harte prompts) ===
  [base-harte] Prompt 1/250 done — JSD=0.2786, 2g=0.0000, 3g=0.3750, 4g=0.9333
  [base-harte] Prompt 2/250 done — JSD=0.4457, 2g=1.0000, 3g=0.0000, 4g=0.0000
  [base-harte] Prompt 3/250 done — JSD=0.2763, 2g=0.0000, 3g=0.2500, 4g=1.0000
  [base-harte] Prompt 4/250 done — JSD=0.4421, 2g=0.0000, 3g=0.0000, 4g=0.0000
  [base-harte] Prompt 5/250 done — JSD=0.3534, 2g=0.2941, 3g=1.0000, 4g=1.0000
  [base-harte] Prompt 6/250 done — JSD=0.4963, 2g=0.3636, 3g=0.9524, 4g=1.0000
  [base-harte] Prompt 7/250 done — JSD=0.4382, 2g=0.3750, 3g=0.8571, 4g=1.0000
  [base-harte] Prompt 8/250 done — JSD=0.3211, 2g=0.2500, 3g=0.5455, 4g=1.0000
  [base-harte] Prompt 9/250 done — JSD=0.3771, 2g=1.0000, 3g=0.0000, 4g=0.0000
  [base-harte] Prompt 10/250 done — JSD=0.3080, 2g=0.0000, 3g=0.0000, 4g=0.0000
  [base-harte] Prompt 11/250 done — JSD=0.3771, 2g=1.0000, 3g=0.0000, 4g=0.0000
  [base-harte] Prompt 12/250 done — JSD=0.3539, 2g=0.8571, 3g=0.9231, 4g=1.0000
  [base-harte]

### 8. Results Summary

In [24]:
print(f"{'Model':<14} {'JSD':>8} {'2-gram':>8} {'3-gram':>8} {'4-gram':>8} {'N':>5}")
print("-" * 57)
for r in [base_harte_results, base_abc_results, harte_results, abc_results]:
    print(
        f"{r['model']:<14} "
        f"{r['jsd_mean']:>8.4f} "
        f"{r['bigram_novelty']:>8.4f} "
        f"{r['trigram_novelty']:>8.4f} "
        f"{r['fourgram_novelty']:>8.4f} "
        f"{r['n_scored']:>5}"
    )

print("""
Interpreting results:
  JSD closer to 0  → chord distribution matches training data more closely (better grammar learning)
  N-gram novelty   → higher = more original progressions not seen in training data

  base-harte : base GPT-2 prompted in Harte notation, scored against Harte dataset
  base-abc   : base GPT-2 prompted in ABC notation,   scored against ABC dataset
  harte      : Harte fine-tuned model
  abc        : ABC fine-tuned model
""")

Model               JSD   2-gram   3-gram   4-gram     N
---------------------------------------------------------
base-harte       0.4033   0.3968   0.5404   0.6576   250
base-abc         0.4279   0.4998   0.6805   0.7817   250
harte            0.4070   0.1624   0.4536   0.6755   250
abc              0.4202   0.0696   0.3201   0.5879   250

Interpreting results:
  JSD closer to 0  → chord distribution matches training data more closely (better grammar learning)
  N-gram novelty   → higher = more original progressions not seen in training data

  base-harte : base GPT-2 prompted in Harte notation, scored against Harte dataset
  base-abc   : base GPT-2 prompted in ABC notation,   scored against ABC dataset
  harte      : Harte fine-tuned model
  abc        : ABC fine-tuned model

